# 05 · Ungrounded vs. RAG-grounded explanations

For the same image and heatmap summary, asks the LLM for a forensic explanation twice: once from the heatmap alone (ungrounded) and once with retrieved forensic-literature passages it must cite (grounded). Shows them side by side with the automatic hallucination checks.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 (not P100)
- **Internet:** On
- **Inputs:** output of **01_extract_faces**; output of **02_train**
- **Secrets (Add-ons → Secrets):** one of `ANTHROPIC_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY` or `LLM_API_KEY` (none needed for Ollama)

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Uses the repo if it is attached as an input (zip uploaded as a Kaggle dataset, or an earlier
# notebook's output); otherwise clones it. For a private repo add a GITHUB_TOKEN secret.
import os, sys, glob, shutil, subprocess

PROJECT = "/kaggle/working/project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
    if found:
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
        if r.returncode:
            raise RuntimeError("git clone failed: attach the project as a dataset or add a GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
print("project ready:", sorted(os.listdir(PROJECT)))

In [ ]:
from src.kaggle_utils import setup, setup_llm, start_ollama
setup(data=True, checkpoint=True)

# --- LLM backend -------------------------------------------------------------------------
LLM_PROVIDER = "anthropic"   # anthropic | openrouter | ollama | openai (any OpenAI-compatible server)
LLM_MODEL = None             # None = provider default; e.g. "meta-llama/llama-3.3-70b-instruct" on OpenRouter
if LLM_PROVIDER == "ollama":
    start_ollama(LLM_MODEL or "llama3.1:8b")   # local model running inside this Kaggle session
assert setup_llm(LLM_PROVIDER, LLM_MODEL), "Add your API key under Add-ons → Secrets"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

from src.config import DATA_DIR
from src.eval.metrics import faithfulness_iou, hallucination_flags
from src.explain.corpus import load_corpus
from src.explain.llm import ForensicExplainer
from src.xai.gradcam import Explainer, draw_bbox

cam, llm = Explainer(), ForensicExplainer()
m = pd.read_csv(DATA_DIR / "manifest.csv")
test = m[m.split == "test"]

## The retrieval corpus

In [ ]:
display(pd.DataFrame([{"id": d.id, "artifact": d.artifact_type, "regions": ", ".join(d.regions), "source": d.citation}
                      for d in load_corpus()]))

## Side-by-side examples
One fake per manipulation method plus one real frame.

In [ ]:
def card(out, exp):
    if out.error:
        return f"**error:** {out.error}"
    flags = [k for k, v in hallucination_flags(out, exp).items() if v and k != "any_hallucination"]
    return (f"**region:** {out.claimed_region} · **artifact:** `{out.artifact_type}` · "
            f"**IoU:** {faithfulness_iou(out.claimed_region, exp.heatmap):.2f}  \n{out.explanation}  \n"
            f"citations: {out.citations or '—'} · flags: **{', '.join(flags) or 'none'}**")

picks = [test[test.method == mth].sample(1, random_state=3).iloc[0] for mth in sorted(test.method.unique())]
for r in picks:
    img = Image.open(DATA_DIR / r.split / r.label / r.file)
    exp = cam(img)
    fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))
    axes[0].imshow(img.resize((224, 224))); axes[1].imshow(draw_bbox(exp.overlay, exp.bbox))
    for a in axes: a.axis("off")
    plt.suptitle(f"{r.method} · predicted {exp.label} ({exp.prob_fake:.2f})", fontsize=9); plt.show()
    u, g = llm.explain(exp, grounded=False), llm.explain(exp, grounded=True)
    display(Markdown(f"*Heatmap summary:* {exp.describe()}\n\n*Retrieved:* {g.retrieved}\n\n"
                     f"| Ungrounded | Grounded (RAG) |\n|---|---|\n| {card(u, exp)} | {card(g, exp)} |"))